In [0]:
# Import Required Libraries
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
def log(msg: str):
    from datetime import datetime
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

In [0]:
def enable_cdf(table_name: str):
    spark.sql(f"""
        ALTER TABLE {table_name}
        SET TBLPROPERTIES(delta.enableChangeDataFeed = true)          
    """)
    log(f"🔁 CDF enable on {table_name}")

In [0]:
def get_row_count(table_name: str) -> int:
    return spark.table(table_name).count()

In [0]:
def table_exists(table_name: str) -> bool:
    try:
        spark.sql(f"DESCRIBE TABLE {table_name}")
        return True
    except Exception:
        return False

In [0]:
def write_delta(
    df: DataFrame,
    table_name: str,
    mode: str = "overwrite",
    partition_by: list = None,
    cdf: bool = True
):
    """
    Write a DataFrame to a Delta table in Unity Catalog.

    Args:
        df           : DataFrame to write
        table_name   : Full table name (catalog.schema.table)
        mode         : 'overwrite' or 'append' (default: overwrite)
        partition_by : List of column names to partition by (optional)
        cdf          : Enable Change Data Feed (default: True)
    """
    
    log(f"Writing to {table_name} [mode={mode}]...")
    write = df.write.format("delta").mode(mode).option("overwriteSchema", "true")
    
    if partition_by:
        write = write.partitionBy(partition_by)
    write.saveAsTable(table_name)

    # Enable CDF after table is written
    if cdf:
        enable_cdf(table_name)

    count = get_row_count(table_name)
    log(f"✅ Done. {count:,} rows written to {table_name}")
        